In [1]:
#pip install scvelo

In [2]:
#pip install python-igraph

In [3]:
#pip install louvain

In [4]:
#pip install scanpy[leiden]

In [5]:
#pip install gseapy

In [6]:
import pandas as pd
import scanpy as sc
import numpy as np
from numpy import linalg as la
from scipy.sparse import issparse
import loompy
import anndata
import scvelo as scv
scv.logging.print_version()

KeyboardInterrupt: 

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display, HTML

import dynamo as dyn
dyn.configuration.set_figure_params('dynamo', background='white')
dyn.get_all_dependencies_version()

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
#import os
#os.chdir('C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated')
#adata_11 = dyn.read('normal-CIS_integrated_subcluster_dynamo.h5ad')

In [ ]:
#adata_NC = adata_11[adata_11.obs['batch'].isin(['Normal','CIS'])]

In [ ]:
#dyn.pl.streamline_plot(adata_NC, color=['louvain'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire')

In [ ]:
#del adata_NC.uns['batch_colors']

In [ ]:
#adata_NC.write('Normal_and_CIS_from_integrated.h5ad', compression='gzip')

In [ ]:
# starting from Normal_and_CIS integrated file with laptop 
import os
os.chdir('/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo')

adata_NC = sc.read('Normal_and_CIS_from_integrated.h5ad')

In [ ]:
adata = adata_NC

In [ ]:
adata.var_names_make_unique()
adata # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`

In [ ]:
# mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('mt-') 
# ribosomal genes
adata.var['ribo'] = adata.var_names.str.startswith(("Rps","Rpl"))
# hemoglobin genes.
adata.var['hb'] = adata.var_names.str.contains(("^hb[^(P)]"))

adata.var

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo','hb'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', color="batch")

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_ribo', color="batch")

In [ ]:
mito_genes = adata.var_names.str.startswith('mt-')
# for each cell compute fraction of counts in mito genes vs. all genes
# the `.A1` is only necessary as X is sparse (to transform to a dense array after summing)
adata.obs['percent_mt2'] = np.sum(
    adata[:, mito_genes].X, axis=1).A1 / np.sum(adata.X, axis=1).A1
# add the total counts per cell as observations-annotation to adata
adata.obs['n_counts'] = adata.X.sum(axis=1).A1

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

print(adata.n_obs, adata.n_vars)

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
# filter for percent mito
adata = adata[adata.obs['pct_counts_mt'] < 10, :]

# filter for percent ribo > 0.05
adata = adata[adata.obs['pct_counts_ribo'] > 5, :]

print("Remaining cells %d"%adata.n_obs)

In [ ]:
adata

In [ ]:
adata_New8 = adata[adata.obs['louvain'].isin(['New_8'])]
adata_New8

In [ ]:
adata_New4 = adata[adata.obs['louvain'].isin(['New_4'])]
adata_New4

In [ ]:
adata_Epi2 = adata[adata.obs['louvain'].isin(['Epi_2'])]
adata_Epi2

In [ ]:
adata_New6 = adata[adata.obs['louvain'].isin(['New_6'])]
adata_New6

In [ ]:
adata_New7 = adata[adata.obs['louvain'].isin(['New_7'])]
adata_New7

In [ ]:
sc.pl.umap(adata_New8, color=['batch'], save='umap_New8_Normal_CIS.pdf')

In [ ]:
sc.pl.umap(adata_New4, color=['batch'], save='umap_New4_Normal_CIS.pdf')

In [ ]:
sc.pl.umap(adata_Epi2, color=['batch'], save='umap_Epi2_Normal_CIS.pdf')

In [ ]:
sc.pl.umap(adata_New6, color=['batch'])

In [ ]:
sc.pl.umap(adata_New7, color=['batch'])

In [ ]:
sc.settings.verbosity = 2  # reduce the verbosity

In [ ]:
sc.tl.rank_genes_groups(adata_New8, 'batch', method='wilcoxon', key_added='wilcoxon')
sc.pl.rank_genes_groups(adata_New8, n_genes=25, sharey=False, key='wilcoxon')

In [ ]:
import seaborn as sns

genes1 = sc.get.rank_genes_groups_df(adata_New8, group='CIS', key='wilcoxon')['names'][:5]
genes2 = sc.get.rank_genes_groups_df(adata_New8, group='Normal', key='wilcoxon')['names'][:5]
genes = genes1.tolist() +  genes2.tolist() 
df = sc.get.obs_df(adata_New8, genes + ['louvain','batch'], use_raw=False)
df2 = df.melt(id_vars=["louvain",'batch'], value_vars=genes)

In [ ]:
sns.catplot(x = "louvain", y = "value", hue = "batch", kind = 'violin', 
               col = "variable", data = df2, col_wrap=4, inner=None)

In [ ]:
import gseapy

In [ ]:
# Gene Set Analysis (Enrichr)
#Available databases : ‘Human’, ‘Mouse’, ‘Yeast’, ‘Fly’, ‘Fish’, ‘Worm’ 
gene_set_names = gseapy.get_library_name(organism='Mouse')
print(gene_set_names)

In [ ]:
#?gseapy.enrichr
glist8 = sc.get.rank_genes_groups_df(adata_New8, group='CIS', 
                                    key='wilcoxon', log2fc_min=0.25, 
                                    pval_cutoff=0.01)['names'].squeeze().str.strip().tolist()
print(len(glist8))

In [ ]:
enr_res_8_1 = gseapy.enrichr(gene_list=glist8,
                     organism='Mouse',
                     gene_sets='GO_Biological_Process_2021',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_8_1.results.head()

In [ ]:
enr_res_8_2 = gseapy.enrichr(gene_list=glist8,
                     organism='Mouse',
                     gene_sets='KEGG_2019_Mouse',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_8_2.results.head()

In [ ]:
gseapy.barplot(enr_res_8_1.res2d,title='GO_Biological_Process_2021')
gseapy.barplot(enr_res_8_1.res2d,title='GO_Biological_Process_2021', ofname='New8_enrichr_GOBP.pdf')

In [ ]:
gseapy.barplot(enr_res_8_1.res2d,title='GO_Biological_Process_2021', top_term=20)

In [ ]:
gseapy.barplot(enr_res_8_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise')
gseapy.barplot(enr_res_8_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', ofname='New8_enrichr_KEGG.pdf')

In [ ]:
gseapy.barplot(enr_res_8_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', top_term=20)

In [ ]:
sc.tl.rank_genes_groups(adata_New7, 'batch', method='wilcoxon', key_added='wilcoxon')
sc.pl.rank_genes_groups(adata_New7, n_genes=25, sharey=False, key='wilcoxon')
adata_New7

In [ ]:
import seaborn as sns

genes1 = sc.get.rank_genes_groups_df(adata_New7, group='CIS', key='wilcoxon')['names'][:5]
genes2 = sc.get.rank_genes_groups_df(adata_New7, group='Normal', key='wilcoxon')['names'][:5]
genes = genes1.tolist() +  genes2.tolist() 
df = sc.get.obs_df(adata, genes + ['louvain','batch'], use_raw=False)
df2 = df.melt(id_vars=["louvain",'batch'], value_vars=genes)

In [ ]:
sns.catplot(x = "louvain", y = "value", hue = "batch", kind = 'violin', 
               col = "variable", data = df2, col_wrap=4, inner=None)

In [ ]:
#?gseapy.enrichr
glist7 = sc.get.rank_genes_groups_df(adata_New7, group='CIS', 
                                    key='wilcoxon', log2fc_min=0.25, 
                                    pval_cutoff=0.01)['names'].squeeze().str.strip().tolist()
print(len(glist7))

In [ ]:
enr_res_3 = gseapy.enrichr(gene_list=glist7,
                     organism='Mouse',
                     gene_sets='GO_Biological_Process_2021',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_3.results.head()

In [ ]:
enr_res_4 = gseapy.enrichr(gene_list=glist7,
                     organism='Mouse',
                     gene_sets='KEGG_2019_Mouse',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_4.results.head()

In [ ]:
gseapy.barplot(enr_res_3.res2d,title='GO_Biological_Process_2021')
gseapy.barplot(enr_res_3.res2d,title='GO_Biological_Process_2021', ofname='New7_enrichr_GOBP.pdf')

In [ ]:
gseapy.barplot(enr_res_3.res2d,title='GO_Biological_Process_2021', top_term=20)

In [ ]:
gseapy.barplot(enr_res_4.res2d,title='KEGG_2019_Mouse', color='darkturquoise')
gseapy.barplot(enr_res_4.res2d,title='KEGG_2019_Mouse', color='darkturquoise', ofname='New7_enrichr_KEGG.pdf')

In [ ]:
gseapy.barplot(enr_res_4.res2d,title='KEGG_2019_Mouse', color='darkturquoise', top_term=20)

In [ ]:
sc.tl.rank_genes_groups(adata_New6, 'batch', method='wilcoxon', key_added='wilcoxon')
sc.pl.rank_genes_groups(adata_New6, n_genes=25, sharey=False, key='wilcoxon')

In [ ]:
import seaborn as sns

genes1 = sc.get.rank_genes_groups_df(adata_New6, group='CIS', key='wilcoxon')['names'][:5]
genes2 = sc.get.rank_genes_groups_df(adata_New6, group='Normal', key='wilcoxon')['names'][:5]
genes = genes1.tolist() +  genes2.tolist() 
df = sc.get.obs_df(adata, genes + ['louvain','batch'], use_raw=False)
df2 = df.melt(id_vars=["louvain",'batch'], value_vars=genes)

In [ ]:
sns.catplot(x = "louvain", y = "value", hue = "batch", kind = 'violin', 
               col = "variable", data = df2, col_wrap=4, inner=None)

In [ ]:
#?gseapy.enrichr
glist6 = sc.get.rank_genes_groups_df(adata_New6, group='CIS', 
                                    key='wilcoxon', log2fc_min=0.25, 
                                    pval_cutoff=0.05)['names'].squeeze().str.strip().tolist()
print(len(glist6))

In [ ]:
enr_res_1 = gseapy.enrichr(gene_list=glist,
                     organism='Mouse',
                     gene_sets='GO_Biological_Process_2021',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_1.results.head()

In [ ]:
enr_res_2 = gseapy.enrichr(gene_list=glist,
                     organism='Mouse',
                     gene_sets='KEGG_2019_Mouse',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_2.results.head()

In [ ]:
gseapy.barplot(enr_res_1.res2d,title='GO_Biological_Process_2021')
gseapy.barplot(enr_res_1.res2d,title='GO_Biological_Process_2021', ofname='New8_enrichr_GOBP.pdf')

In [ ]:
gseapy.barplot(enr_res_1.res2d,title='GO_Biological_Process_2021', top_term=20)

In [ ]:
gseapy.barplot(enr_res_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise')
gseapy.barplot(enr_res_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', ofname='New8_enrichr_KEGG.pdf')

In [ ]:
gseapy.barplot(enr_res_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', top_term=20)

In [ ]:
sc.tl.rank_genes_groups(adata_New4, 'batch', method='wilcoxon', key_added='wilcoxon')
sc.pl.rank_genes_groups(adata_New4, n_genes=25, sharey=False, key='wilcoxon')

In [ ]:
import seaborn as sns

genes1 = sc.get.rank_genes_groups_df(adata_New4, group='CIS', key='wilcoxon')['names'][:5]
genes2 = sc.get.rank_genes_groups_df(adata_New4, group='Normal', key='wilcoxon')['names'][:5]
genes = genes1.tolist() +  genes2.tolist() 
df = sc.get.obs_df(adata, genes + ['louvain','batch'], use_raw=False)
df2 = df.melt(id_vars=["louvain",'batch'], value_vars=genes)

In [ ]:
sns.catplot(x = "louvain", y = "value", hue = "batch", kind = 'violin', 
               col = "variable", data = df2, col_wrap=4, inner=None)

In [ ]:
#?gseapy.enrichr
glist4 = sc.get.rank_genes_groups_df(adata_New4, group='CIS', 
                                    key='wilcoxon', log2fc_min=0.25, 
                                    pval_cutoff=0.01)['names'].squeeze().str.strip().tolist()
print(len(glist4))

In [ ]:
enr_res_4_1 = gseapy.enrichr(gene_list=glist4,
                     organism='Mouse',
                     gene_sets='GO_Biological_Process_2021',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_4_1.results.head()

In [ ]:
enr_res_4_2 = gseapy.enrichr(gene_list=glist4,
                     organism='Mouse',
                     gene_sets='KEGG_2019_Mouse',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_4_2.results.head()

In [ ]:
gseapy.barplot(enr_res_4_1.res2d,title='GO_Biological_Process_2021')
gseapy.barplot(enr_res_4_1.res2d,title='GO_Biological_Process_2021', ofname='New4_enrichr_GOBP.pdf')

In [ ]:
gseapy.barplot(enr_res_4_1.res2d,title='GO_Biological_Process_2021', top_term=20)

In [ ]:
gseapy.barplot(enr_res_4_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise')
gseapy.barplot(enr_res_4_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', ofname='New4_enrichr_KEGG.pdf')

In [ ]:
gseapy.barplot(enr_res_4_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', top_term=20)

In [ ]:
sc.tl.rank_genes_groups(adata_Epi2, 'batch', method='wilcoxon', key_added='wilcoxon')
sc.pl.rank_genes_groups(adata_Epi2, n_genes=25, sharey=False, key='wilcoxon')

In [ ]:
import seaborn as sns

genes1 = sc.get.rank_genes_groups_df(adata_Epi2, group='CIS', key='wilcoxon')['names'][:5]
genes2 = sc.get.rank_genes_groups_df(adata_Epi2, group='Normal', key='wilcoxon')['names'][:5]
genes = genes1.tolist() +  genes2.tolist() 
df = sc.get.obs_df(adata, genes + ['louvain','batch'], use_raw=False)
df2 = df.melt(id_vars=["louvain",'batch'], value_vars=genes)

In [ ]:
sns.catplot(x = "louvain", y = "value", hue = "batch", kind = 'violin', 
               col = "variable", data = df2, col_wrap=4, inner=None)

In [ ]:
#?gseapy.enrichr
glist2 = sc.get.rank_genes_groups_df(adata_Epi2, group='CIS', 
                                    key='wilcoxon', log2fc_min=0.25, 
                                    pval_cutoff=0.01)['names'].squeeze().str.strip().tolist()
print(len(glist2))

In [ ]:
enr_res_2_1 = gseapy.enrichr(gene_list=glist2,
                     organism='Mouse',
                     gene_sets='GO_Biological_Process_2021',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_2_1.results.head()

In [ ]:
enr_res_2_2 = gseapy.enrichr(gene_list=glist2,
                     organism='Mouse',
                     gene_sets='KEGG_2019_Mouse',
                     description='pathway',
                     cutoff = 0.5)

In [ ]:
enr_res_2_2.results.head()

In [ ]:
gseapy.barplot(enr_res_2_1.res2d,title='GO_Biological_Process_2021')
gseapy.barplot(enr_res_2_1.res2d,title='GO_Biological_Process_2021', ofname='Epi2_enrichr_GOBP.pdf')

In [ ]:
gseapy.barplot(enr_res_2_1.res2d,title='GO_Biological_Process_2021', top_term=20)

In [ ]:
gseapy.barplot(enr_res_2_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise')
gseapy.barplot(enr_res_2_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', ofname='Epi2_enrichr_KEGG.pdf')

In [ ]:
gseapy.barplot(enr_res_2_2.res2d,title='KEGG_2019_Mouse', color='darkturquoise', top_term=20)

In [ ]:
#######################
###### GSEA ###########
#######################

In [ ]:
gene_rank8 = sc.get.rank_genes_groups_df(adata_New8, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank8.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)
gene_rank8

In [ ]:
# calculate_qc_metrics will calculate number of cells per gene
sc.pp.calculate_qc_metrics(adata_New8, percent_top=None, log1p=False, inplace=True)

# filter for genes expressed in at least 30 cells.
gene_rank8 = gene_rank8[gene_rank8['names'].isin(adata_New8.var_names[adata_New8.var.n_cells_by_counts>20])]

In [ ]:
gene_rank8

In [ ]:
#Available databases : ‘Human’, ‘Mouse’, ‘Yeast’, ‘Fly’, ‘Fish’, ‘Worm’ 
gene_set_names = gseapy.get_library_name(organism ='Mouse')
print(gene_set_names)

In [ ]:
res8 = gseapy.prerank(rnk=gene_rank8, gene_sets='HDSigDB_Mouse_2021', processes=8, 
                     permutation_num=100, # reduce number to speed up testing
                     outdir='/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo', 
                      format='pdf', seed=6, min_size=2)

In [ ]:
terms = res8.res2d.index
terms[:20]

In [ ]:
res8.res2d

In [ ]:
res8.res2d.sort_values(by=['pval'], inplace=True, ascending=True)
res8.res2d[:20]

In [ ]:
gseapy.gseaplot(rank_metric=res8.ranking, term=terms[0], **res8.results[terms[0]])

In [ ]:
res8_2 = gseapy.prerank(rnk=gene_rank8, gene_sets='GO_Biological_Process_2021', processes=8, 
                     permutation_num=100, # reduce number to speed up testing
                     outdir='/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo', 
                      format='pdf', seed=6, min_size=2)

In [ ]:
terms = res8_2.res2d.index
terms[:20]

In [ ]:
res8_2.res2d


In [ ]:
res8_2.res2d.sort_values(by=['pval'], inplace=True, ascending=True)
res8_2.res2d

In [ ]:
gseapy.gseaplot(rank_metric=res8_2.ranking, term=terms[0], **res8.results[terms[0]])

In [ ]:
gene_rank7 = sc.get.rank_genes_groups_df(adata_New7, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank7.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)
gene_rank7

In [ ]:
res7 = gseapy.prerank(rnk=gene_rank7, gene_sets='GO_Biological_Process_2021', processes=8, 
                     permutation_num=100, # reduce number to speed up testing
                     outdir='/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo', format='pdf', seed=6, min_size=2)

In [ ]:
terms = res7.res2d.index
terms[:20]

In [ ]:
res7.res2d[:20]

In [ ]:
res7.res2d.sort_values(by=['pval'], inplace=True, ascending=True)
res7.res2d

In [ ]:
gseapy.gseaplot(rank_metric=res7.ranking, term=terms[0], **res6.results[terms[0]])

In [ ]:
gseapy.gseaplot(rank_metric=res7.ranking, term='regulation of complement activation (GO:0030449)', **res6.results[terms[0]])

In [ ]:
gene_rank6 = sc.get.rank_genes_groups_df(adata_New6, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank6.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)
gene_rank6

In [ ]:
res6 = gseapy.prerank(rnk=gene_rank6, gene_sets='GO_Biological_Process_2021', processes=8, 
                     permutation_num=100, # reduce number to speed up testing
                     outdir='/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo', format='pdf', seed=6, min_size=2)

In [ ]:
terms = res6.res2d.index
terms[:20]

In [ ]:
res6.res2d[:20]

In [ ]:
res6.res2d.sort_values(by=['pval'], inplace=True, ascending=True)
res6.res2d

In [ ]:
gseapy.gseaplot(rank_metric=res6.ranking, term=terms[0], **res6.results[terms[0]])

In [ ]:
gseapy.gseaplot(rank_metric=res6.ranking, term='regulation of complement activation (GO:0030449)', **res6.results[terms[0]])

In [ ]:
gene_rank4 = sc.get.rank_genes_groups_df(adata_New4, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank4.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)
gene_rank4

In [ ]:
res4 = gseapy.prerank(rnk=gene_rank4, gene_sets='GO_Biological_Process_2021', processes=8, 
                     permutation_num=100, # reduce number to speed up testing
                     outdir='/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo', format='pdf', seed=6, min_size=2)

In [ ]:
terms = res4.res2d.index
terms[:20]

In [ ]:
res4.res2d[:20]

In [ ]:
res4.res2d.sort_values(by=['pval'], inplace=True, ascending=True)
res4.res2d

In [ ]:
gseapy.gseaplot(rank_metric=res4.ranking, term=terms[0], **res4.results[terms[0]])

In [ ]:
gseapy.gseaplot(rank_metric=res4.ranking, term='regulation of complement activation (GO:0030449)', **res4.results[terms[0]])

In [ ]:
gene_rank2 = sc.get.rank_genes_groups_df(adata_Epi2, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank2.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)
gene_rank2

In [ ]:
res2 = gseapy.prerank(rnk=gene_rank2, gene_sets='GO_Biological_Process_2021', processes=8, 
                     permutation_num=100, # reduce number to speed up testing
                     outdir='/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo', format='pdf', seed=6, min_size=2)

In [ ]:
terms = res2.res2d.index
terms[:20]

In [ ]:
res2.res2d[:20]

In [ ]:
res2.res2d.sort_values(by=['pval'], inplace=True, ascending=True)
res2.res2d

In [ ]:
gseapy.gseaplot(rank_metric=res4.ranking, term=terms[0], **res4.results[terms[0]])

In [ ]:
gseapy.gseaplot(rank_metric=res4.ranking, term='regulation of complement activation (GO:0030449)', **res4.results[terms[0]])

In [ ]:
sc.tl.rank_genes_groups(adata, 'batch', method='wilcoxon', key_added='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, key='wilcoxon')

In [ ]:
gene_rank = sc.get.rank_genes_groups_df(adata, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)
gene_rank

In [ ]:
# calculate_qc_metrics will calculate number of cells per gene
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)

# filter for genes expressed in at least 30 cells.
gene_rank = gene_rank[gene_rank['names'].isin(adata.var_names[adata.var.n_cells_by_counts>20])]

In [ ]:
gene_rank

In [ ]:
gene_rank = gene_rank.rename(columns={'names': '0', 'logfoldchanges':'1'})
gene_rank

In [ ]:
res = gseapy.prerank(rnk=gene_rank, gene_sets='GO_Biological_Process_2021', processes=8, 
                     permutation_num=100, # reduce number to speed up testing
                     outdir='/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo', 
                     format='pdf', seed=6, min_size=2)

In [ ]:
terms = res.res2d.index
terms[:20]

In [ ]:
res.res2d

In [ ]:
res.res2d.sort_values(by=['pval'], inplace=True, ascending=True)
res.res2d

In [ ]:
gseapy.gseaplot(rank_metric=res.ranking, term=terms[0], **res.results[terms[0]])

In [ ]:
sc.tl.rank_genes_groups(adata_New8, 'batch', method='wilcoxon', key_added='wilcoxon')
gene_rank8 = sc.get.rank_genes_groups_df(adata_New8, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank8.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)

# calculate_qc_metrics will calculate number of cells per gene
sc.pp.calculate_qc_metrics(adata_New8, percent_top=None, log1p=False, inplace=True)

# filter for genes expressed in at least 30 cells.
gene_rank8 = gene_rank8[gene_rank8['names'].isin(adata_New8.var_names[adata_New8.var.n_cells_by_counts>5])]
gene_rank8 = gene_rank8.rename(columns={'names': '0', 'logfoldchanges':'1'})
gene_rank8
gene_rank8.to_csv('gene_rank8_for_gsea.csv')

In [ ]:
sc.tl.rank_genes_groups(adata_New7, 'batch', method='wilcoxon', key_added='wilcoxon')
gene_rank7 = sc.get.rank_genes_groups_df(adata_New7, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank7.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)

# calculate_qc_metrics will calculate number of cells per gene
sc.pp.calculate_qc_metrics(adata_New7, percent_top=None, log1p=False, inplace=True)

# filter for genes expressed in at least 30 cells.
gene_rank7 = gene_rank7[gene_rank7['names'].isin(adata_New7.var_names[adata_New7.var.n_cells_by_counts>5])]
gene_rank7 = gene_rank7.rename(columns={'names': '0', 'logfoldchanges':'1'})
gene_rank7
gene_rank7.to_csv('gene_rank7_for_gsea.csv')

In [ ]:
sc.tl.rank_genes_groups(adata_New6, 'batch', method='wilcoxon', key_added='wilcoxon')
gene_rank6 = sc.get.rank_genes_groups_df(adata_New6, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank6.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)

# calculate_qc_metrics will calculate number of cells per gene
sc.pp.calculate_qc_metrics(adata_New6, percent_top=None, log1p=False, inplace=True)

# filter for genes expressed in at least 30 cells.
gene_rank6 = gene_rank6[gene_rank6['names'].isin(adata_New6.var_names[adata_New6.var.n_cells_by_counts>5])]
gene_rank6 = gene_rank6.rename(columns={'names': '0', 'logfoldchanges':'1'})
gene_rank6
gene_rank6.to_csv('gene_rank6_for_gsea.csv')

In [ ]:
sc.tl.rank_genes_groups(adata_New4, 'batch', method='wilcoxon', key_added='wilcoxon')
gene_rank4 = sc.get.rank_genes_groups_df(adata_New4, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank4.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)

# calculate_qc_metrics will calculate number of cells per gene
sc.pp.calculate_qc_metrics(adata_New4, percent_top=None, log1p=False, inplace=True)

# filter for genes expressed in at least 30 cells.
gene_rank4 = gene_rank4[gene_rank4['names'].isin(adata_New4.var_names[adata_New4.var.n_cells_by_counts>5])]
gene_rank4 = gene_rank4.rename(columns={'names': '0', 'logfoldchanges':'1'})
gene_rank4
gene_rank4.to_csv('gene_rank4_for_gsea.csv')

In [ ]:
sc.tl.rank_genes_groups(adata_Epi2, 'batch', method='wilcoxon', key_added='wilcoxon')
gene_rank2 = sc.get.rank_genes_groups_df(adata_Epi2, group='CIS', key='wilcoxon')[['names','logfoldchanges']]
gene_rank2.sort_values(by=['logfoldchanges'], inplace=True, ascending=False)

# calculate_qc_metrics will calculate number of cells per gene
sc.pp.calculate_qc_metrics(adata_Epi2, percent_top=None, log1p=False, inplace=True)

# filter for genes expressed in at least 30 cells.
gene_rank2 = gene_rank2[gene_rank2['names'].isin(adata_Epi2.var_names[adata_Epi2.var.n_cells_by_counts>5])]
gene_rank2 = gene_rank2.rename(columns={'names': '0', 'logfoldchanges':'1'})
gene_rank2
gene_rank2.to_csv('gene_rank2_for_gsea.csv')